# QLoRA fine-tuning — Gemma 4 E2B (Hungarian text→JSON extraction)

Thin Colab wrapper around `src/train.py` (dev-plan step 3.4). All the actual
logic (data loading, model/LoRA setup, `SFTTrainer` config) lives in the
script, not here — this notebook just installs Unsloth, fetches the repo +
data, and invokes the CLI.

**Runtime:** Runtime → Change runtime type → T4 GPU, before running anything below.

In [ ]:
%%capture
!pip install unsloth

## Repo

Clone the repo (set `REPO_URL` below) so `src/` is available.

In [ ]:
REPO_URL = "https://github.com/<your-username>/hu-llm-finetune-lora.git"  # noqa: set this
REPO_DIR = "hu-llm-finetune-lora"

!git clone "$REPO_URL" "$REPO_DIR"
%cd $REPO_DIR

## Synthetic training data

`data/synthetic/` is gitignored (regenerable, LLM-generated — see `CLAUDE.md`),
so it doesn't come with the clone. Upload the three domain JSONL files
(`medical.jsonl`, `business.jsonl`, `technology.jsonl`) produced locally by
`python src/generate_data.py`, e.g. via a zip:

In [ ]:
from google.colab import files
import zipfile

uploaded = files.upload()  # select a zip of data/synthetic/*.jsonl
for name in uploaded:
    if name.endswith(".zip"):
        with zipfile.ZipFile(name) as zf:
            zf.extractall("data/synthetic")

## Smoke test

Quick end-to-end shakeout (~60 steps) before committing to a full run.

In [11]:
!python src/train.py --smoke --out outputs/smoke

Loading base model unsloth/gemma-4-E2B-it (4-bit QLoRA)...
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias 

## Full training run

In [12]:
!python src/train.py --out outputs/adapter

Loading base model unsloth/gemma-4-E2B-it (4-bit QLoRA)...
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias 

The trained LoRA adapter + tokenizer are saved to `outputs/adapter/`.
Download it (or push it to the Hugging Face Hub — step 3.7, not part of this
notebook) before the Colab runtime recycles.

In [13]:
import shutil
from google.colab import files

shutil.make_archive("adapter", "zip", "outputs/adapter")
files.download("adapter.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [15]:
import shutil
shutil.copytree(
    "/content/hu-llm-finetune-lora/outputs/adapter",
    "/content/drive/MyDrive/llm-finetune-lora/adapter",
    dirs_exist_ok=True,
)

'/content/drive/MyDrive/llm-finetune-lora/adapter'